In [32]:
import numpy as np
import pandas as pd
from datetime import timedelta

# === INPUT FILES (cambiali se hanno nomi diversi) ===
BINANCE_PATH = "binance_ALL_1h_730d.csv"
GDELT_HOURLY_PATH = "gdelt_hourly_2Y_top5.csv"

# === OUTPUT FILES ===
OUT_ENRICHED_HOURLY = "enriched_hourly_2y.csv"
OUT_DQ_REPORT       = "dq_report_enrichment_2y.csv"

# === PARAMETRI PULIZIA ===
TONE_ABS_MAX = 50


In [33]:
#Helper (ore attese)
def to_utc_hour(x):
    return pd.to_datetime(x, utc=True, errors="coerce").dt.floor("h")

def expected_hours(start, end):
    # ore attese in [start, end)
    return pd.date_range(start=start, end=end - timedelta(hours=1), freq="h", tz="UTC")

def dq_coverage(df, start, end, label):
    exp = expected_hours(start, end)
    out = []
    for asset, g in df.groupby("asset"):
        pres = pd.Index(g["ts_hour_utc"].dropna().unique())
        miss = pd.Index(exp).difference(pres)
        out.append({
            "dataset": label,
            "asset": asset,
            "expected_hours": int(len(exp)),
            "present_hours": int(len(pres)),
            "missing_hours": int(len(miss)),
            "coverage": float(len(pres) / len(exp)) if len(exp) else None,
            "missing_examples_first10": ";".join([str(x) for x in miss[:10]])
        })
    return pd.DataFrame(out).sort_values(["dataset","asset"])

def to_utc_hour(x):
    return pd.to_datetime(x, utc=True, errors="coerce").dt.floor("h")

def expected_hours(start, end):
    # ore attese in [start, end)
    return pd.date_range(start=start, end=end - timedelta(hours=1), freq="h", tz="UTC")

def dq_coverage(df, start, end, label):
    exp = expected_hours(start, end)
    out = []
    for asset, g in df.groupby("asset"):
        pres = pd.Index(g["ts_hour_utc"].dropna().unique())
        miss = pd.Index(exp).difference(pres)
        out.append({
            "dataset": label,
            "asset": asset,
            "expected_hours": int(len(exp)),
            "present_hours": int(len(pres)),
            "missing_hours": int(len(miss)),
            "coverage": float(len(pres) / len(exp)) if len(exp) else None,
            "missing_examples_first10": ";".join([str(x) for x in miss[:10]])
        })
    return pd.DataFrame(out).sort_values(["dataset","asset"])

In [34]:
#Load + standardizzazione BINANCE (curated)
b = pd.read_csv(BINANCE_PATH)

# timestamp chiave
b["open_time"] = pd.to_datetime(b["open_time"], utc=True, errors="coerce")
b["ts_hour_utc"] = b["open_time"].dt.floor("h")

# asset
b["asset"] = b["symbol"].astype(str).str.replace("USDT", "", regex=False)

# numerici (incluso n_trades)
for c in ["open","high","low","close","volume","n_trades"]:
    if c in b.columns:
        b[c] = pd.to_numeric(b[c], errors="coerce")

# selezione colonne utili
keep_b = ["asset","ts_hour_utc","open","high","low","close","volume"]
if "n_trades" in b.columns:
    keep_b.append("n_trades")

b = b[keep_b].copy()

# dedup chiave naturale
b = (b.dropna(subset=["asset","ts_hour_utc"])
       .sort_values(["asset","ts_hour_utc"])
       .drop_duplicates(["asset","ts_hour_utc"], keep="last")
       .reset_index(drop=True))

print("BINANCE rows:", len(b))
print("BINANCE dup key:", b.duplicated(["asset","ts_hour_utc"]).sum())
b.head()


BINANCE rows: 87605
BINANCE dup key: 0


,asset,ts_hour_utc,open,high,low,close,volume,n_trades
0,BNB,2024-01-25 00:00:00+00:00,293.0,293.8,291.5,292.9,29616.679,9170
1,BNB,2024-01-25 01:00:00+00:00,292.9,293.2,290.8,291.0,38681.631,11134
2,BNB,2024-01-25 02:00:00+00:00,291.0,291.4,287.5,288.2,50254.384,15827
3,BNB,2024-01-25 03:00:00+00:00,288.2,291.3,288.0,290.5,32296.050,8451
4,BNB,2024-01-25 04:00:00+00:00,290.4,290.6,288.2,288.8,20793.320,7182


In [35]:
#Load + standardizzazione GDELT HOURLY (curated)
g = pd.read_csv(GDELT_HOURLY_PATH)

# chiave oraria
g["ts_hour_utc"] = to_utc_hour(g["ts_hour_utc"])

# numerici
g["news_count_hour"] = pd.to_numeric(g["news_count_hour"], errors="coerce").fillna(0)
g["avg_tone_hour"]   = pd.to_numeric(g["avg_tone_hour"], errors="coerce")

# regole tone
g.loc[g["news_count_hour"] == 0, "avg_tone_hour"] = np.nan
g.loc[g["avg_tone_hour"].abs() > TONE_ABS_MAX, "avg_tone_hour"] = np.nan

# opzionale share
extra = []
if "news_share_hour" in g.columns:
    g["news_share_hour"] = pd.to_numeric(g["news_share_hour"], errors="coerce")
    extra.append("news_share_hour")

keep_g = ["asset","ts_hour_utc","news_count_hour","avg_tone_hour"] + extra
g = g[keep_g].copy()

# dedup chiave naturale
g = (g.dropna(subset=["asset","ts_hour_utc"])
       .sort_values(["asset","ts_hour_utc"])
       .drop_duplicates(["asset","ts_hour_utc"], keep="last")
       .reset_index(drop=True))

print("GDELT rows:", len(g))
print("GDELT dup key:", g.duplicated(["asset","ts_hour_utc"]).sum())
g.head()


GDELT rows: 84905
GDELT dup key: 0


,asset,ts_hour_utc,news_count_hour,avg_tone_hour,news_share_hour
0,BNB,2024-01-25 00:00:00+00:00,2,-2.8011,0.0162
1,BNB,2024-01-25 01:00:00+00:00,3,-2.6148,0.0284
2,BNB,2024-01-25 02:00:00+00:00,3,-1.4714,0.0253
3,BNB,2024-01-25 03:00:00+00:00,10,-1.4407,0.0820
4,BNB,2024-01-25 04:00:00+00:00,1,-1.5385,0.0135


In [36]:
#Allinea range (intersezione) + DQ pre-join
start = max(b["ts_hour_utc"].min(), g["ts_hour_utc"].min())
end   = min(b["ts_hour_utc"].max() + timedelta(hours=1),
            g["ts_hour_utc"].max() + timedelta(hours=1))

print("INTERSECTION RANGE:", start, "->", end)

b2 = b[(b["ts_hour_utc"] >= start) & (b["ts_hour_utc"] < end)].copy()
g2 = g[(g["ts_hour_utc"] >= start) & (g["ts_hour_utc"] < end)].copy()

dq_pre = pd.concat([
    dq_coverage(b2, start, end, "BINANCE_pre"),
    dq_coverage(g2, start, end, "GDELT_pre")
], ignore_index=True)

dq_pre

INTERSECTION RANGE: 2024-01-25 00:00:00+00:00 -> 2026-01-24 00:00:00+00:00


,dataset,asset,expected_hours,present_hours,missing_hours,coverage,missing_examples_first10
0,BINANCE_pre,BNB,17520,17520,0,1.000000,
1,BINANCE_pre,BTC,17520,17520,0,1.000000,
2,BINANCE_pre,ETH,17520,17520,0,1.000000,
3,BINANCE_pre,SOL,17520,17520,0,1.000000,
4,BINANCE_pre,XRP,17520,17520,0,1.000000,
5,GDELT_pre,BNB,17520,16981,539,0.969235,2024-01-28 22:00:00+00:00;2024-02-04 03:00:00+...
6,GDELT_pre,BTC,17520,16981,539,0.969235,2024-01-28 22:00:00+00:00;2024-02-04 03:00:00+...
7,GDELT_pre,ETH,17520,16981,539,0.969235,2024-01-28 22:00:00+00:00;2024-02-04 03:00:00+...
8,GDELT_pre,SOL,17520,16981,539,0.969235,2024-01-28 22:00:00+00:00;2024-02-04 03:00:00+...
9,GDELT_pre,XRP,17520,16981,539,0.969235,2024-01-28 22:00:00+00:00;2024-02-04 03:00:00+...


In [37]:
#Crea “time spine” completa (1 riga per ora × asset)
assets = sorted(b2["asset"].unique())
hours = expected_hours(start, end)

grid = pd.MultiIndex.from_product([assets, hours], names=["asset","ts_hour_utc"]).to_frame(index=False)
print("GRID rows:", len(grid))
grid.head()

GRID rows: 87600


,asset,ts_hour_utc
0,BNB,2024-01-25 00:00:00+00:00
1,BNB,2024-01-25 01:00:00+00:00
2,BNB,2024-01-25 02:00:00+00:00
3,BNB,2024-01-25 03:00:00+00:00
4,BNB,2024-01-25 04:00:00+00:00


In [38]:
#Integration (join su chiave naturale) + flag hole GDELT
# Binance su grid
en = grid.merge(b2, on=["asset","ts_hour_utc"], how="left", validate="one_to_one")

# GDELT su grid
en = en.merge(g2, on=["asset","ts_hour_utc"], how="left", validate="one_to_one")

# hole GDELT (distinguo “0 news” vs “mancante”)
en["gdelt_missing"] = en["news_count_hour"].isna() & en["avg_tone_hour"].isna()
en["news_count_hour"] = en["news_count_hour"].fillna(0)

print("EN rows:", len(en))
print("EN dup key:", en.duplicated(["asset","ts_hour_utc"]).sum())
en.head()


EN rows: 87600
EN dup key: 0


,asset,ts_hour_utc,open,high,low,close,volume,n_trades,news_count_hour,avg_tone_hour,news_share_hour,gdelt_missing
0,BNB,2024-01-25 00:00:00+00:00,293.0,293.8,291.5,292.9,29616.679,9170,2.0,-2.8011,0.0162,False
1,BNB,2024-01-25 01:00:00+00:00,292.9,293.2,290.8,291.0,38681.631,11134,3.0,-2.6148,0.0284,False
2,BNB,2024-01-25 02:00:00+00:00,291.0,291.4,287.5,288.2,50254.384,15827,3.0,-1.4714,0.0253,False
3,BNB,2024-01-25 03:00:00+00:00,288.2,291.3,288.0,290.5,32296.050,8451,10.0,-1.4407,0.0820,False
4,BNB,2024-01-25 04:00:00+00:00,290.4,290.6,288.2,288.8,20793.320,7182,1.0,-1.5385,0.0135,False


In [39]:
#Enrichment PREZZI (returns, range, volatilità rolling)
en = en.sort_values(["asset","ts_hour_utc"]).reset_index(drop=True)

# log close e return 1h
en["log_close"] = np.log(en["close"].replace(0, np.nan))
en["ret_1h"] = en.groupby("asset")["log_close"].diff(1)

# oscillazione intrahour (high-low su open)
en["hl_range_pct"] = (en["high"] - en["low"]) / en["open"].replace(0, np.nan)

en[["asset","ts_hour_utc","close","log_close","ret_1h","hl_range_pct"]].head(10)

,asset,ts_hour_utc,close,log_close,ret_1h,hl_range_pct
0,BNB,2024-01-25 00:00:00+00:00,292.9,5.679831,NaN,0.007850
1,BNB,2024-01-25 01:00:00+00:00,291.0,5.673323,-0.006508,0.008194
2,BNB,2024-01-25 02:00:00+00:00,288.2,5.663655,-0.009669,0.013402
3,BNB,2024-01-25 03:00:00+00:00,290.5,5.671604,0.007949,0.011450
4,BNB,2024-01-25 04:00:00+00:00,288.8,5.665734,-0.005869,0.008264
5,BNB,2024-01-25 05:00:00+00:00,288.7,5.665388,-0.000346,0.005540
6,BNB,2024-01-25 06:00:00+00:00,290.2,5.670570,0.005182,0.006581
7,BNB,2024-01-25 07:00:00+00:00,292.1,5.677096,0.006526,0.009993
8,BNB,2024-01-25 08:00:00+00:00,291.9,5.676411,-0.000685,0.004451
9,BNB,2024-01-25 09:00:00+00:00,289.5,5.668155,-0.008256,0.010620


In [40]:
# Totali ret 1 h nan
print("close NaN:", en["close"].isna().sum())
print("ret_1h NaN:", en["ret_1h"].isna().sum())

# Per asset
nan_by_asset = en.groupby("asset")[["close","ret_1h"]].apply(lambda x: x.isna().sum())
print(nan_by_asset)

close NaN: 0
ret_1h NaN: 5
       close  ret_1h
asset               
BNB        0       1
BTC        0       1
ETH        0       1
SOL        0       1
XRP        0       1


In [41]:
#Enrichment NEWS (spike P95 SEMESTRALE)

# label semestre (H1/H2) per ogni timestamp
en["semester"] = (
    en["ts_hour_utc"].dt.year.astype(str)
    + "-H"
    + np.where(en["ts_hour_utc"].dt.month <= 6, "1", "2")
).astype("string")

# P95 per asset + semestre (sulla news_count_hour)
en["news_p95"] = en.groupby(["asset", "semester"])["news_count_hour"].transform(lambda s: s.quantile(0.95))

# spike
en["news_spike"] = (en["news_count_hour"] >= en["news_p95"]).astype(int)
en.drop(columns=["semester"], inplace=True)
en[["asset","ts_hour_utc","news_count_hour","news_spike","avg_tone_hour","news_p95"]].head(10)


,asset,ts_hour_utc,news_count_hour,news_spike,avg_tone_hour,news_p95
0,BNB,2024-01-25 00:00:00+00:00,2.0,0,-2.8011,6.0
1,BNB,2024-01-25 01:00:00+00:00,3.0,0,-2.6148,6.0
2,BNB,2024-01-25 02:00:00+00:00,3.0,0,-1.4714,6.0
3,BNB,2024-01-25 03:00:00+00:00,10.0,1,-1.4407,6.0
4,BNB,2024-01-25 04:00:00+00:00,1.0,0,-1.5385,6.0
5,BNB,2024-01-25 05:00:00+00:00,1.0,0,0.8969,6.0
6,BNB,2024-01-25 06:00:00+00:00,3.0,0,-0.3413,6.0
7,BNB,2024-01-25 07:00:00+00:00,1.0,0,3.3514,6.0
8,BNB,2024-01-25 08:00:00+00:00,2.0,0,-0.8301,6.0
9,BNB,2024-01-25 09:00:00+00:00,0.0,0,NaN,6.0


In [42]:
#DQ post (coverage + percentuale hole GDELT)
dq_post = dq_coverage(en[["asset","ts_hour_utc"]], start, end, "ENRICHED_post")

holes = (en.groupby("asset")["gdelt_missing"].mean()
           .rename("pct_gdelt_missing")
           .reset_index())

dq_final = dq_post.merge(holes, on="asset", how="left")
dq_final


,dataset,asset,expected_hours,present_hours,missing_hours,coverage,missing_examples_first10,pct_gdelt_missing
0,ENRICHED_post,BNB,17520,17520,0,1.0,,0.030765
1,ENRICHED_post,BTC,17520,17520,0,1.0,,0.030765
2,ENRICHED_post,ETH,17520,17520,0,1.0,,0.030765
3,ENRICHED_post,SOL,17520,17520,0,1.0,,0.030765
4,ENRICHED_post,XRP,17520,17520,0,1.0,,0.030765


In [43]:
# Salva hourly + DQ report (senza daily dataset)

en.to_csv(OUT_ENRICHED_HOURLY, index=False)
dq_final.to_csv(OUT_DQ_REPORT, index=False)

print("Saved:", OUT_ENRICHED_HOURLY)
print("Saved:", OUT_DQ_REPORT)


Saved: enriched_hourly_2y.csv
Saved: dq_report_enrichment_2y.csv
